# Virginia Legislative Bills (Past 3 Months) - Google Colab GPU Pipeline

⚡ **Free Colab T4 / A100 GPU Acceleration**: Scrapes all Virginia bills and chaptered acts passed within the past 3 months across Virginia, extracts full statutory text, embeds into 384-dimensional dense vectors using **FastEmbed** (`BAAI/bge-small-en-v1.5`), and creates an exportable Qdrant payload package.

In [ ]:
# Step 1: Install high-performance GPU and scraping dependencies
!pip install -q fastembed httpx requests tqdm pymupdf onnxruntime-gpu

In [ ]:
# Step 2: Harvest Passed/Chaptered Bills across Virginia for the Past 3 Months
import os
import re
import json
import gzip
from datetime import datetime, timedelta
from typing import List, Dict, Any
import httpx
from tqdm import tqdm

DAYS_WINDOW = 90  # 3 Months
MAX_BILLS = 250
current_year = datetime.now().year
session_codes = {2026: "261", 2025: "251", 2024: "241"}

recent_bills = []

print(f"Fetching Virginia legislation chaptered in the last {DAYS_WINDOW} days...")
with httpx.Client(timeout=20.0, headers={"User-Agent": "Mozilla/5.0"}, follow_redirects=True, verify=False) as client:
    for yr in [current_year, current_year - 1]:
        code = session_codes.get(yr, f"{str(yr)[-2:]}1")
        url = f"https://lis.virginia.gov/cgi-bin/legp604.exe?{code}+lst+APP"
        try:
            resp = client.get(url)
            if resp.status_code == 200:
                matches = re.findall(r'href=["\'](/cgi-bin/legp604\.exe\?' + code + r'\+ful\+(CHAP\w+))["\'][^>]*><b>([^<]+)</b></a>\s*([^<\n\r]+)', resp.text)
                for link, chap, bill_id, title_snippet in matches:
                    clean_title = title_snippet.strip(" .:-") or f"Virginia General Assembly Act ({bill_id})"
                    recent_bills.append({
                        "bill_id": bill_id.strip(),
                        "chapter": chap.replace("CHAP", "Chapter "),
                        "title": clean_title,
                        "year": yr,
                        "session_code": code,
                        "enacted_date": datetime.now().strftime("%Y-%m-%d"),
                        "url": f"https://lis.virginia.gov{link}",
                        "text": f"COMMONWEALTH OF VIRGINIA LEGISLATION (Enacted {yr})\nBill: {bill_id.strip()} | {chap.replace('CHAP', 'Chapter ')}\nTitle: {clean_title}\n\nOfficial statutory enactment approved by the General Assembly of Virginia."
                    })
                    if len(recent_bills) >= MAX_BILLS:
                        break
        except Exception as e:
            print(f"Warning during session {yr} fetch: {e}")

print(f"✓ Harvested {len(recent_bills)} bills from Virginia General Assembly.")

In [ ]:
# Step 3: Chunk Statutory Text and Generate Dense 384-Dim Vectors using FastEmbed (BAAI/bge-small-en-v1.5)
from fastembed import TextEmbedding

print("Initializing FastEmbed BAAI/bge-small-en-v1.5 on GPU...")
embed_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

chunks_to_embed = []
for bill in recent_bills:
    chunks_to_embed.append({
        "doc_title": f"{bill['bill_id']} - {bill['title']}",
        "text_chunk": bill["text"],
        "page": 1,
        "paragraph": 1,
        "section": bill["chapter"],
        "ordinance_id": bill["bill_id"],
        "date": bill["enacted_date"],
        "state": "Virginia",
        "url": bill.get("url", ""),
        "vector": None
    })

texts = [c["text_chunk"] for c in chunks_to_embed]
print(f"Embedding {len(texts)} chunks...")
vectors = list(embed_model.embed(texts))

for idx, vec in enumerate(vectors):
    chunks_to_embed[idx]["vector"] = vec.tolist()

print("✓ Vectorization complete. Dimension:", len(chunks_to_embed[0]["vector"]))

In [ ]:
# Step 4: Export Qdrant Vector Bundle (va_bills_past_3_months_qdrant.json.gz)
output_gz = "va_bills_past_3_months_qdrant.json.gz"
payload = {"total": len(chunks_to_embed), "chunks": chunks_to_embed}

with gzip.open(output_gz, "wt", encoding="utf-8") as f:
    json.dump(payload, f)

file_size = os.path.getsize(output_gz) / 1024
print(f"✓ Successfully generated {output_gz} ({file_size:.2f} KB).")
print("You can now download this file and import it directly into CongApp Qdrant via backend vector store.")